In [2]:
# Imports
import os
import sys
import subprocess
import matplotlib

from pathlib import Path

## aggregateBatteryOutput.py
Script for aggregate battery outputs in intervals.

In [ ]:
def aggregateBatteryOutput():
    tool = (Path(os.environ["SUMO_HOME"]) / "tools/output/aggregateBatteryOutput.py").resolve()
    result = subprocess.run([
        sys.executable,
        str(tool),
        "-i", r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\electric_bus_2026-07-29-15-34-21_battery.xml",
        "-o", r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\eBuS\files\batteryAggregatedx.xml",
        "-t", "600",
    ], check=True)

    print("aggregateBatteryOutput finished.")

aggregateBatteryOutput()

Started


## plotXMLAttributes.py
Create multiple 2D-plots of 2 arbitrary attributes from on or more xml files aggregated by an third attribute (i.e. detector-id).

In [ ]:
def plotXMLAttributes():
    tool = (Path(os.environ["SUMO_HOME"]) / "tools/visualization/plotXMLAttributes.py").resolve()
    battery_file = r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\electric_bus_2026-07-29-16-07-33_battery.xml"

    # Plot 1: energy consumed per bus over time
    subprocess.run([
        sys.executable, str(tool),
        "-x", "timestep_time",
        "-y", "vehicle_energyConsumed",
        "--idattr", "vehicle_id",
        "--xlabel", "time [s]",
        "--ylabel", "energy consumed [Wh]",
        "--title", "Energy consumed per bus",
        "-o", "../files/plots/energyConsumed.png",
        battery_file,
    ], check=True)

    # Plot 2: energy charged per bus over time
    subprocess.run([
        sys.executable, str(tool),
        "-x", "timestep_time",
        "-y", "vehicle_energyCharged",
        "--idattr", "vehicle_id",
        "--xlabel", "time [s]",
        "--ylabel", "energy charged [Wh]",
        "--title", "Energy charged per bus",
        "-o", "../files/plots/energyCharged.png",
        battery_file,
    ], check=True)

    print("plotXMLAttributes finished.")

plotXMLAttributes()

## tripStatistics.py

This script is to calculate the global performance indices according to SUMO-based simulation results. The calculation functions are directly defined in this script.

In [ ]:
def tripStatistics():

    tool = (Path(os.environ["SUMO_HOME"]) / "tools/output/tripStatistics.py").resolve()
    result = subprocess.run([
        "python",
        f"{str(tool)}",
        "-t", r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\electric_bus_2026-07-29-10-34-54_tripinfo.xml",
        "-o", "../files/tripinfo.txt",
        "-e"
    ])
    print("tripStatistics finished.")

tripStatistics()

tripStatistics finished.


## computeStoppingPlaceUsage.py
This tool reads stop-output and tracks the number of stopped vehicles over time at stopping places (i.e. parkingArea). A distinct output file will be created for each stopping place.

In [ ]:
def computeStoppingPlaceUsage():

    tool = (Path(os.environ["SUMO_HOME"]) / "tools/output/computeStoppingPlaceUsage.py").resolve()
    result = subprocess.run([
        sys.executable,
        f"{str(tool)}",
        "-t", r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\electric_bus_2026-07-29-16-07-33_stopinfo.xml",
        "-o", "../files/tripinfo.txt",
        "-e"
    ])
    print("computeStoppingPlaceUsage finished.")

computeStoppingPlaceUsage()

## plot_trajectories.py
Create plot of all trajectories obtained from a file generated through --fcd-output. This tool in particular is located in <SUMO_HOME>/tools.

In [18]:
def plot_trajectories():

    tool = (Path(os.environ["SUMO_HOME"]) / "tools/plot_trajectories.py").resolve()
    result = subprocess.run([
        sys.executable,
        f"{str(tool)}",
        "-t", "xy",
        "-o", "../files/plots/allLocations_output.png",
        r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\electric_bus_2026-07-29-16-07-33_fcdinfo.xml",
        "--scatterplot"
    ])
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        print("STDOUT:", result.stdout)
        print("Return code:", result.returncode)
    else:
        print("plot_trajectories finished.")

plot_trajectories()

plot_trajectories finished.


## plotStops.py
Plot a public transport schedule (either planned or actual timings). 

In [26]:
import os
import re
import subprocess
import sys
import xml.etree.ElementTree as ET
from pathlib import Path


def _get_first_veh_id(route_file):
    """Return the id of the first vehicle/trip/flow found in a SUMO route file."""
    for _, elem in ET.iterparse(route_file, events=("start",)):
        if elem.tag in ("route") and "id" in elem.attrib:
            return elem.attrib["id"]
    raise ValueError(f"No vehicle/trip/flow with an id found in {route_file}")


def plotStops():
    tool = (Path(os.environ["SUMO_HOME"]) / "tools/visualization/plotStops.py").resolve()

    route_file = r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\electric\e_routes.rou.xml"
    stops_file = r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\berlin_bus_stops.add.xml"

    veh_id = _get_first_veh_id(route_file)

    result = subprocess.run([
        sys.executable,
        str(tool),
        "-r", route_file,
        "-a", stops_file,
        "-i", veh_id,
        "-v",
        "--legend",
        "--filter-ids", "*",
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print("STDERR:", result.stderr)
        print("STDOUT:", result.stdout)
        print("Return code:", result.returncode)
    else:
        print("plotStops finished.")


plotStops()


STDERR: Error: could not find vehicle trip or flow with id 'cicerostrasse_3001_route' in route-file 'C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\electric\e_routes.rou.xml'

STDOUT: 
Return code: 1


## plot_tripinfo_distributions.py
plot_tripinfo_distributions.py reads one or multiple tripinfo-files and plots a selected measure (attribute of the read tripinfo-files). The measure is visualised as vertical bars that represent the numbers of occurrences of the measure (vehicles) that fall into a bin.

**SEEMS DEPRECATED WITHOUT NOTICE**

In [ ]:
def plot_tripinfo_distributions():
    tool = (Path(os.environ["SUMO_HOME"]) / "tools/purgatory/plot_tripinfo_distributions.py").resolve()

    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"  # Headless Backend für subprocess

    result = subprocess.run([
        "uv", "run", 
        "--python", "3.11",
        "--with", "matplotlib",
        str(tool),
        "-i", r"C:\Users\svens\Documents\FU-Berlin\BeST-eBuS\best-ebus\scenario\sumo\output\electric_bus_2026-07-29-10-34-54_tripinfo.xml",
        "-o", "../files/plots/stopCountDist.png",
        "--measure", "waitingCount",
        "--bins", "10",
        "--maxV", "50",
        "--xlabel", "number of stops [-]",
        "--ylabel", "count [-]",
        "--title", "distribution of number of stops",
        "--colors", "blue",
        "-b", 
        "--no-legend"
    ], capture_output=True, text=True)
    
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        print("STDOUT:", result.stdout)
    else:
        print("plot_tripinfo_distributions finished.")

plot_tripinfo_distributions()